<a href="https://colab.research.google.com/github/vaino118/Security-Analytics_S2_2026---Practical-Labs-Lab-1-Lab-2-Lab-3-/blob/main/Lab_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# LAB 2 - PART B: ACCESS ANALYTICS AND BEHAVIOURAL BASELINING
# Google Colab Version - Upload Required
# ============================================

# Step 1: Install required libraries (if needed)
!pip install pandas numpy matplotlib seaborn -q

# Step 2: Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import zipfile
import io
import os
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("="*60)
print("LAB 2 - PART B: ACCESS ANALYTICS")
print("="*60)
print("\nPlease upload the required data files when prompted.\n")

# ============================================
# STEP 3: UPLOAD AND LOAD DATA FILES
# ============================================

def upload_and_load_files():
    """
    Function to handle file uploads in Google Colab
    """
    print("\n--- FILE UPLOAD INSTRUCTIONS ---")
    print("1. Click the 'Choose Files' button below")
    print("2. Select ALL the CSV files from 'Lab 2 Data Files.zip'")
    print("3. You can upload multiple files at once (Ctrl+Click or Shift+Click)")
    print("\nRequired files:")
    print("  ✓ ofs_employee_directory.csv")
    print("  ✓ ofs_access_policy_matrix.csv")
    print("  ✓ ofs_vpn_authentication_logs.csv")
    print("  ✓ ofs_endpoint_process_logs.csv")
    print("  ✓ ofs_file_access_logs.csv")
    print("  ✓ ofs_proxy_dlp_logs.csv")
    print("  ✓ ofs_access_session_training.csv")
    print("  ✓ ofs_access_session_investigation.csv")
    print("\n⚠️  IMPORTANT: You can also upload the ZIP file and I'll extract it!")
    print("-"*60)

    uploaded = files.upload()

    # Check if ZIP file was uploaded
    zip_files = [f for f in uploaded.keys() if f.endswith('.zip')]

    if zip_files:
        print(f"\n📦 ZIP file detected: {zip_files[0]}")
        print("Extracting files...")

        # Extract the ZIP file
        with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
            zip_ref.extractall('.')
        print("✅ ZIP file extracted successfully!")

        # Now check for CSV files
        csv_files = [f for f in os.listdir('.') if f.endswith('.csv')]
    else:
        # Use uploaded files directly
        csv_files = [f for f in uploaded.keys() if f.endswith('.csv')]

    # Define expected files
    expected_files = {
        'employee_dir': 'ofs_employee_directory.csv',
        'access_policy': 'ofs_access_policy_matrix.csv',
        'vpn_logs': 'ofs_vpn_authentication_logs.csv',
        'process_logs': 'ofs_endpoint_process_logs.csv',
        'file_logs': 'ofs_file_access_logs.csv',
        'proxy_logs': 'ofs_proxy_dlp_logs.csv',
        'training_data': 'ofs_access_session_training.csv',
        'investigation_data': 'ofs_access_session_investigation.csv'
    }

    dataframes = {}

    print("\n📂 Loading files...")
    for name, filename in expected_files.items():
        try:
            # Check if file exists
            if filename in csv_files or os.path.exists(filename):
                dataframes[name] = pd.read_csv(filename)
                print(f"  ✓ {filename} loaded ({len(dataframes[name])} rows)")
            else:
                print(f"  ✗ {filename} NOT FOUND")
        except Exception as e:
            print(f"  ✗ Error loading {filename}: {e}")

    return dataframes

# Load the data
data = upload_and_load_files()

# Verify all files loaded
if len(data) < 8:
    print("\n⚠️  Some files are missing. Please ensure all CSV files are uploaded.")
    print("\n💡 TIP: You can upload the ZIP file instead of individual CSVs!")
else:
    print("\n✅ All files loaded successfully!")
    print("-"*60)

# ============================================
# STEP 4: ASSIGN VARIABLES (if data loaded successfully)
# ============================================

if len(data) >= 8:
    employee_dir = data['employee_dir']
    access_policy = data['access_policy']
    vpn_logs = data['vpn_logs']
    process_logs = data['process_logs']
    file_logs = data['file_logs']
    proxy_logs = data['proxy_logs']
    training_data = data['training_data']
    investigation_data = data['investigation_data']

    print("\n✅ Data loaded successfully!")
    print(f"\nData Summary:")
    print(f"  • Employee Directory: {len(employee_dir)} rows")
    print(f"  • Access Policy: {len(access_policy)} rows")
    print(f"  • VPN Logs: {len(vpn_logs)} rows")
    print(f"  • Process Logs: {len(process_logs)} rows")
    print(f"  • File Logs: {len(file_logs)} rows")
    print(f"  • Proxy Logs: {len(proxy_logs)} rows")
    print(f"  • Training Data: {len(training_data)} rows")
    print(f"  • Investigation Data: {len(investigation_data)} rows")

    # ============================================
    # B1: REPRODUCIBLE PREPARATION (3 marks)
    # ============================================
    print("\n" + "="*60)
    print("B1: REPRODUCIBLE PREPARATION")
    print("="*60)

    # Convert timestamps
    def convert_timestamp(df, col='timestamp'):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')
        return df

    # Apply to all datasets
    vpn_logs = convert_timestamp(vpn_logs)
    process_logs = convert_timestamp(process_logs)
    file_logs = convert_timestamp(file_logs)
    proxy_logs = convert_timestamp(proxy_logs)
    training_data['session_start'] = pd.to_datetime(training_data['session_start'])
    investigation_data['session_start'] = pd.to_datetime(investigation_data['session_start'])

    # Create summary statistics
    datasets = {
        'Employee Directory': employee_dir,
        'Access Policy': access_policy,
        'VPN Logs': vpn_logs,
        'Process Logs': process_logs,
        'File Logs': file_logs,
        'Proxy Logs': proxy_logs,
        'Training Data': training_data,
        'Investigation Data': investigation_data
    }

    summary_list = []
    for name, df in datasets.items():
        summary_list.append({
            'Dataset': name,
            'Rows': len(df),
            'Columns': len(df.columns),
            'Missing Values': df.isnull().sum().sum(),
            'Duplicate Events': df.duplicated().sum()
        })

    summary_df = pd.DataFrame(summary_list)
    print("\n📊 Data Summary Statistics:")
    print(summary_df.to_string(index=False))

    # Create derived fields
    print("\n🔧 Creating Derived Fields...")

    if 'timestamp' in vpn_logs.columns:
        vpn_logs['hour'] = vpn_logs['timestamp'].dt.hour
        vpn_logs['weekend'] = vpn_logs['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)
        vpn_logs['off_hours'] = ((vpn_logs['hour'] < 6) | (vpn_logs['hour'] > 20)).astype(int)
        print("  ✓ VPN logs: hour, weekend, off_hours")

    if 'timestamp' in file_logs.columns:
        file_logs['hour'] = file_logs['timestamp'].dt.hour
        file_logs['weekend'] = file_logs['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)
        file_logs['download_mb'] = file_logs['bytes_transferred'] / (1024 * 1024)
        file_logs['sensitivity_binary'] = file_logs['sensitivity'].apply(
            lambda x: 1 if x in ['Restricted', 'Confidential'] else 0
        )
        print("  ✓ File logs: hour, weekend, download_mb, sensitivity_binary")

    if 'timestamp' in proxy_logs.columns:
        proxy_logs['hour'] = proxy_logs['timestamp'].dt.hour
        proxy_logs['weekend'] = proxy_logs['timestamp'].dt.dayofweek.isin([5, 6]).astype(int)
        proxy_logs['upload_mb'] = proxy_logs['bytes_out'] / (1024 * 1024)
        print("  ✓ Proxy logs: hour, weekend, upload_mb")

    print("✅ Derived fields created successfully!")

    # ============================================
    # B2: USER AND PEER-GROUP BASELINES (5 marks)
    # ============================================
    print("\n" + "="*60)
    print("B2: USER AND PEER-GROUP BASELINES")
    print("="*60)

    # Visualisation 1: Failed Authentications by Department
    print("\n📊 Visualisation 1: Failed Authentications by Department")

    vpn_with_dept = vpn_logs.merge(
        employee_dir[['user_id', 'department']],
        on='user_id',
        how='left'
    )

    dept_failures = vpn_with_dept[vpn_with_dept['result'] == 'FAILURE'].groupby('department').size().reset_index(name='failures')

    plt.figure(figsize=(10, 6))
    bars = plt.bar(dept_failures['department'], dept_failures['failures'], color=['#368ccb', '#7ca5c5', '#dc3913', '#ffbc00', '#5cb85c', '#f0ad4e', '#d9534f', '#337ab7'])
    plt.title('Failed Authentications by Department', fontsize=16, fontweight='bold')
    plt.xlabel('Department', fontsize=12)
    plt.ylabel('Number of Failed Logins', fontsize=12)
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height, f'{int(height)}', ha='center', va='bottom')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

    print("\n📝 Interpretation:")
    print("• Question: Which departments have the highest rates of failed authentication attempts?")
    print("• Observation: Customer Service shows the highest number of failed logins ({:.0f}), followed by Information Technology ({:.0f}). This is likely due to the large number of users and high interaction with external systems.".format(
        dept_failures[dept_failures['department']=='Customer Service']['failures'].values[0] if 'Customer Service' in dept_failures['department'].values else 0,
        dept_failures[dept_failures['department']=='Information Technology']['failures'].values[0] if 'Information Technology' in dept_failures['department'].values else 0
    ))
    print("• Operational Significance: This baseline helps identify departments that may be more prone to credential issues or targeted by brute-force attacks.")

    # Visualisation 2: Hourly Login Pattern Comparison
    print("\n📊 Visualisation 2: Hourly Login Pattern - User vs. Peer Group")

    user_vpn = vpn_logs[vpn_logs['user_id'] == 'mhaingura']

    cs_vpn = vpn_logs.merge(
        employee_dir[['user_id', 'department']],
        on='user_id',
        how='left'
    )
    cs_vpn = cs_vpn[cs_vpn['department'] == 'Customer Service']

    plt.figure(figsize=(12, 6))

    if len(cs_vpn) > 0:
        sns.histplot(cs_vpn['hour'], bins=24, alpha=0.6, color='blue',
                     label='Customer Service Peers', kde=False, stat='count')

    if len(user_vpn) > 0:
        sns.histplot(user_vpn['hour'], bins=24, alpha=0.8, color='red',
                     label='User: mhaingura', kde=False, stat='count')

    plt.title('Hourly VPN Activity: User vs. Peer Group Baseline', fontsize=16, fontweight='bold')
    plt.xlabel('Hour of Day (0-23)', fontsize=12)
    plt.ylabel('Number of Logins', fontsize=12)
    plt.legend()
    plt.xticks(range(0, 24, 2))
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("\n📝 Interpretation:")
    print("• Question: Does user 'mhaingura' 's login activity significantly deviate from their department's normal pattern?")
    print("• Observation: The user's session on 2026-09-11 at 18:18 shows a login from South Africa at 18:18, which is outside their normal hours (07:30-17:00) and from a new device.")
    print("• Operational Significance: This is a high-risk anomaly. The authentication event is an outlier when compared to their own historical pattern and their peer group.")

    # Visualisation 3: Sensitive Data Access by Department
    print("\n📊 Visualisation 3: Download Volume by Department and Data Sensitivity")

    if 'department' in file_logs.columns or 'user_id' in file_logs.columns:
        # Merge with employee directory if needed
        if 'department' not in file_logs.columns:
            file_with_dept = file_logs.merge(
                employee_dir[['user_id', 'department']],
                on='user_id',
                how='left'
            )
        else:
            file_with_dept = file_logs

        # Ensure download_mb exists
        if 'download_mb' not in file_with_dept.columns and 'bytes_transferred' in file_with_dept.columns:
            file_with_dept['download_mb'] = file_with_dept['bytes_transferred'] / (1024 * 1024)

        dept_sensitivity = file_with_dept.groupby(['department', 'sensitivity']).agg({
            'download_mb': 'sum'
        }).reset_index()

        dept_sensitivity_pivot = dept_sensitivity.pivot(
            index='department',
            columns='sensitivity',
            values='download_mb'
        ).fillna(0)

        if not dept_sensitivity_pivot.empty:
            dept_sensitivity_pivot.plot(kind='bar', stacked=True, figsize=(12, 6))
            plt.title('Download Volume by Department and Data Sensitivity', fontsize=16, fontweight='bold')
            plt.xlabel('Department', fontsize=12)
            plt.ylabel('Total Download Volume (MB)', fontsize=12)
            plt.xticks(rotation=45, ha='right')
            plt.legend(title='Sensitivity', bbox_to_anchor=(1.05, 1), loc='upper left')
            plt.tight_layout()
            plt.show()

    print("\n📝 Interpretation:")
    print("• Question: Which departments and users are responsible for the largest volumes of data downloads?")
    print("• Observation: The Compliance department has the highest volume of downloads, which is expected given its function.")
    print("• Operational Significance: This demonstrates that while some departments have a legitimate need to access sensitive data, a user within Customer Service is downloading an unusual volume.")

    # Summary Table
    print("\n📊 Summary Table: Anomalous Sessions for User mhaingura")

    anomalous_sessions = investigation_data[
        (investigation_data['new_device'] == 1) |
        (investigation_data['off_hours'] == 1) |
        (investigation_data['privilege_mismatch'] == 1) |
        (investigation_data['impossible_travel'] == 1)
    ]

    mhaingura_anomalies = anomalous_sessions[anomalous_sessions['user_id'] == 'mhaingura']

    if len(mhaingura_anomalies) > 0:
        summary_table = mhaingura_anomalies[['session_id', 'session_start', 'user_id',
                                             'new_device', 'off_hours', 'privilege_mismatch',
                                             'impossible_travel', 'sensitive_resources',
                                             'bytes_downloaded_mb', 'data_upload_mb']].copy()

        # Add descriptions
        descriptions = []
        for idx, row in summary_table.iterrows():
            if 'VPN' in row['session_id']:
                desc = 'Off-hours VPN login from new device/country'
            elif 'KYC' in row['session_id']:
                desc = 'High-volume download of Restricted KYC data'
            elif 'EXFIL' in row['session_id']:
                desc = 'High-volume upload to external site'
            else:
                desc = 'Anomalous session detected'
            descriptions.append(desc)

        summary_table['Description'] = descriptions
        print(summary_table.to_string(index=False))
    else:
        print("No anomalous sessions found for user mhaingura")

    # ============================================
    # B3: TRANSPARENT ACCESS-RISK RULE (4 marks)
    # ============================================
    print("\n" + "="*60)
    print("B3: TRANSPARENT ACCESS-RISK RULE")
    print("="*60)

    def calculate_risk_score(row):
        """
        Calculate risk score based on four indicators:
        - impossible_travel: +40
        - privilege_mismatch: +35
        - off_hours: +15
        - new_device: +10
        """
        score = 0
        score += row['impossible_travel'] * 40
        score += row['privilege_mismatch'] * 35
        score += row['off_hours'] * 15
        score += row['new_device'] * 10
        return score

    # Apply rule
    investigation_data['risk_score'] = investigation_data.apply(calculate_risk_score, axis=1)

    print("\n📋 Risk Score Rules:")
    print("  • Impossible Travel: +40 points")
    print("  • Privilege Mismatch: +35 points")
    print("  • Off-Hours Activity: +15 points")
    print("  • New Device: +10 points")
    print("\n  🚨 Threshold: Risk Score >= 50 = HIGH RISK")

    # Identify high-risk sessions
    high_risk = investigation_data[investigation_data['risk_score'] >= 50].sort_values('risk_score', ascending=False)

    print(f"\n📊 High-risk sessions identified: {len(high_risk)}")

    if len(high_risk) > 0:
        print("\nTop High-Risk Sessions:")
        print(high_risk[['session_id', 'session_start', 'user_id', 'risk_score',
                         'impossible_travel', 'privilege_mismatch', 'off_hours', 'new_device']].to_string(index=False))

        # Export top 15
        high_risk_export = investigation_data[investigation_data['risk_score'] >= 50].head(15)
        high_risk_export[['session_id', 'session_start', 'user_id', 'risk_score']].to_csv(
            'top_15_high_risk_sessions_rule.csv',
            index=False
        )
        print("\n✅ Top 15 high-risk sessions exported to CSV")

        # User risk summary
        user_risk_summary = investigation_data.groupby('user_id').agg({
            'risk_score': 'sum',
            'session_id': 'count'
        }).sort_values('risk_score', ascending=False)

        print("\n👤 User Risk Summary (Top 5):")
        print(user_risk_summary.head(5).to_string())

    # False positives and missed behaviours
    print("\n📝 False Positives and Missed Behaviours:")
    print("\n🔴 Likely False Positive:")
    print("A legitimate power user (e.g., supervisor with CUSTOMER_EXPORT permissions) accessing bulk data")
    print("during off-hours from a new device could be incorrectly flagged.")
    print("Example: Compliance Manager working late from home to review a large dataset.")
    print("This would score 0 (privilege_mismatch) + 15 (off_hours) + 10 (new_device) = 25 (below threshold)")

    print("\n🟡 Likely Missed Behaviour:")
    print('A "low and slow" insider threat would be missed by this rule:')
    print("  • Gradually exfiltrating small amounts of data (e.g., 10 MB per day) over many weeks")
    print("  • Activity occurring during normal working hours")
    print("  • Using their trusted corporate device")
    print("  • Accessing only data within their role's permissions")
    print("\nExample: Loan Officer slowly downloading customer records over several weeks.")
    print("The rule would not flag this as there is no new device, off-hours, privilege mismatch, or impossible travel.")

    print("\n" + "="*60)
    print("✅ PART B COMPLETED SUCCESSFULLY!")
    print("="*60)

else:
    print("\n❌ Some files could not be loaded. Please run the cell again and upload ALL CSV files.")
    print("\n💡 Quick Tip: You can upload the ZIP file instead of individual CSVs!")

LAB 2 - PART B: ACCESS ANALYTICS

Please upload the required data files when prompted.


--- FILE UPLOAD INSTRUCTIONS ---
1. Click the 'Choose Files' button below
2. Select ALL the CSV files from 'Lab 2 Data Files.zip'
3. You can upload multiple files at once (Ctrl+Click or Shift+Click)

Required files:
  ✓ ofs_employee_directory.csv
  ✓ ofs_access_policy_matrix.csv
  ✓ ofs_vpn_authentication_logs.csv
  ✓ ofs_endpoint_process_logs.csv
  ✓ ofs_file_access_logs.csv
  ✓ ofs_proxy_dlp_logs.csv
  ✓ ofs_access_session_training.csv
  ✓ ofs_access_session_investigation.csv

⚠️  IMPORTANT: You can also upload the ZIP file and I'll extract it!
------------------------------------------------------------


In [ ]:
# ============================================
# PART D1 - MODEL DESIGN AND IMPLEMENTATION
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("D1: MODEL DESIGN AND IMPLEMENTATION")
print("="*80)

# ============================================
# 1. LOAD AND EXPLORE TRAINING DATA
# ============================================

print("\n📂 Loading training data...")

training_data = pd.read_csv('ofs_access_session_training.csv')

print(f"Training data shape: {training_data.shape}")
print(f"Columns: {training_data.columns.tolist()}")
print(f"\nClass distribution:")
print(training_data['label'].value_counts())
print(f"Normal (0): {len(training_data[training_data['label'] == 0])}")
print(f"Suspicious (1): {len(training_data[training_data['label'] == 1])}")
print(f"Percentage suspicious: {len(training_data[training_data['label'] == 1])/len(training_data)*100:.1f}%")

# ============================================
# 2. SELECT FEATURES (Based on Data Dictionary)
# ============================================

print("\n📊 Selecting features based on data dictionary...")

# Features selected from the data dictionary with risk rationale
feature_columns = [
    'new_device',           # From data dictionary: "1 if device has not been seen for user"
    'off_hours',            # From data dictionary: "1 if outside user normal hours"
    'failed_logins_30m',    # From data dictionary: "Failed attempts in previous 30 minutes"
    'sensitive_resources',  # From data dictionary: "Count of restricted/confidential resource groups accessed"
    'peer_deviation_score', # From data dictionary: "Deviation from peer group behaviour"
    'process_risk_score',   # From data dictionary: "0-100 process-behaviour risk score"
    'destination_risk_score' # From data dictionary: "0-100 external-destination risk score"
]

print(f"Selected features: {feature_columns}")

# Prepare feature matrix and target vector
X = training_data[feature_columns]
y = training_data['label']

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

# ============================================
# 3. CREATE STRATIFIED 70/30 TRAIN-TEST SPLIT
# ============================================

print("\n🔀 Creating stratified 70/30 train-test split...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,  # Fixed random seed for reproducibility
    stratify=y        # Preserve class distribution
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nTraining set class distribution:")
print(y_train.value_counts())
print(f"\nTest set class distribution:")
print(y_test.value_counts())

# ============================================
# 4. SCALE FEATURES
# ============================================

print("\n📏 Scaling features...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled successfully")

# ============================================
# 5. TRAIN RANDOM FOREST CLASSIFIER
# ============================================

print("\n🌲 Training Random Forest Classifier...")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'  # Handle any class imbalance
)

model.fit(X_train_scaled, y_train)

print("✅ Model trained successfully")

# ============================================
# 6. FEATURE IMPORTANCE
# ============================================

print("\n📊 Feature Importance:")

feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance.to_string(index=False))

# ============================================
# 7. PREDICT ON TEST SET
# ============================================

print("\n🔮 Making predictions on test set...")

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# ============================================
# 8. EVALUATION METRICS
# ============================================

print("\n" + "="*80)
print("MODEL EVALUATION RESULTS")
print("="*80)

# Confusion Matrix
print("\n📊 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\n  True Negatives (Correctly predicted normal): {cm[0,0]}")
print(f"  False Positives (False alarms): {cm[0,1]}")
print(f"  False Negatives (Missed attacks): {cm[1,0]}")
print(f"  True Positives (Correctly detected attacks): {cm[1,1]}")

# Individual Metrics
print("\n📊 Individual Metrics:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
false_negative_rate = cm[1,0] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0

print(f"  Accuracy: {accuracy:.4f}  → Percentage of all predictions that are correct")
print(f"  Precision: {precision:.4f} → When model predicts attack, how often is it correct?")
print(f"  Recall: {recall:.4f}    → How many actual attacks did the model catch?")
print(f"  F1-Score: {f1:.4f}     → Harmonic mean of precision and recall")
print(f"  False Negative Rate: {false_negative_rate:.4f} → Percentage of attacks missed")

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

# ============================================
# 9. SCORE INVESTIGATION DATA
# ============================================

print("\n" + "="*80)
print("SCORING INVESTIGATION DATA")
print("="*80)

# Load investigation data
investigation_data = pd.read_csv('ofs_access_session_investigation.csv')
print(f"Investigation data shape: {investigation_data.shape}")

# Use same features
X_investigation = investigation_data[feature_columns]

# Scale
X_investigation_scaled = scaler.transform(X_investigation)

# Predict probabilities
investigation_data['probability'] = model.predict_proba(X_investigation_scaled)[:, 1]
investigation_data['prediction'] = model.predict(X_investigation_scaled)

# Export top 15 high-risk sessions
def get_interpretation(prob):
    if prob >= 0.9:
        return 'Very High Risk'
    elif prob >= 0.7:
        return 'High Risk'
    elif prob >= 0.5:
        return 'Medium Risk'
    else:
        return 'Low Risk'

investigation_data['interpretation'] = investigation_data['probability'].apply(get_interpretation)

top_15 = investigation_data.nlargest(15, 'probability')

# Select columns for export
export_cols = ['session_id', 'session_start', 'user_id', 'probability', 'interpretation']
top_15[export_cols].to_csv('top_15_high_risk_sessions.csv', index=False)

print("\n✅ Top 15 high-risk sessions exported to 'top_15_high_risk_sessions.csv'")

print("\n📊 Top 15 High-Risk Sessions:")
print(top_15[['session_id', 'user_id', 'probability', 'interpretation']].to_string(index=False))

# ============================================
# 10. SAVE MODEL
# ============================================

import joblib
joblib.dump(model, 'access_risk_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("\n✅ Model saved as 'access_risk_model.pkl'")
print("✅ Scaler saved as 'scaler.pkl'")

print("\n" + "="*80)
print("D1 COMPLETE")
print("="*80)

D1: MODEL DESIGN AND IMPLEMENTATION

📂 Loading training data...
Training data shape: (3000, 22)
Columns: ['session_id', 'session_start', 'user_id', 'department', 'source_country', 'new_device', 'off_hours', 'failed_logins_30m', 'mfa_denials', 'mfa_approvals', 'unique_resources', 'sensitive_resources', 'denied_accesses', 'files_read', 'bytes_downloaded_mb', 'data_upload_mb', 'process_risk_score', 'destination_risk_score', 'peer_deviation_score', 'impossible_travel', 'privilege_mismatch', 'label']

Class distribution:
label
0    2557
1     443
Name: count, dtype: int64
Normal (0): 2557
Suspicious (1): 443
Percentage suspicious: 14.8%

📊 Selecting features based on data dictionary...
Selected features: ['new_device', 'off_hours', 'failed_logins_30m', 'sensitive_resources', 'peer_deviation_score', 'process_risk_score', 'destination_risk_score']

Feature matrix shape: (3000, 7)
Target vector shape: (3000,)

🔀 Creating stratified 70/30 train-test split...
Training set: 2100 samples
Test set:

In [ ]:
# ============================================
# PART C1 - COMPLETE INCIDENT TIMELINE
# WITH CORRECT EMAIL COLUMN NAMES
# ============================================

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("="*120)
print("C1: CORRELATED INCIDENT TIMELINE")
print("Email | VPN | Endpoint | File | Proxy/DLP | User | Asset | Threat Intelligence")
print("="*120)

# ============================================
# RELOAD EMAIL LOGS WITH CORRECT DATA
# ============================================

print("\n📧 Reloading email_logs from CSV...")
email_logs = pd.read_csv('ofs_email_gateway_logs.csv')
print(f"✅ Loaded {len(email_logs)} email events")

# Confirm columns
print(f"Columns: {email_logs.columns.tolist()}")

# Convert timestamps
email_logs['timestamp'] = pd.to_datetime(email_logs['timestamp'], errors='coerce')
print("  ✓ email_logs.timestamp")

# ============================================
# CONVERT OTHER TIMESTAMPS
# ============================================

print("\n🕐 Converting other timestamps...")

for df_name in ['vpn_logs', 'process_logs', 'file_logs', 'proxy_logs']:
    if df_name in locals():
        df = eval(df_name)
        if 'timestamp' in df.columns and len(df) > 0:
            df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
            print(f"  ✓ {df_name}.timestamp")

if 'investigation_data' in locals() and len(investigation_data) > 0:
    if 'session_start' in investigation_data.columns:
        investigation_data['session_start'] = pd.to_datetime(investigation_data['session_start'], errors='coerce')
        print("  ✓ investigation_data.session_start")

if 'threat_intel' in locals() and len(threat_intel) > 0:
    if 'last_updated' in threat_intel.columns:
        threat_intel['last_updated'] = pd.to_datetime(threat_intel['last_updated'], errors='coerce')
        print("  ✓ threat_intel.last_updated")

print("✅ Timestamps converted")

# ============================================
# GET USER AND ASSET INFO
# ============================================

print("\n👤 Getting user and asset information...")

employee_info = employee_dir[employee_dir['user_id'] == 'mhaingura'] if 'employee_dir' in locals() and len(employee_dir) > 0 else pd.DataFrame()
asset_info = asset_inventory[asset_inventory['primary_user'] == 'mhaingura'] if 'asset_inventory' in locals() and len(asset_inventory) > 0 else pd.DataFrame()

if not employee_info.empty:
    print(f"User: {employee_info['employee_name'].values[0]} ({employee_info['user_id'].values[0]})")
    print(f"  Role: {employee_info['role'].values[0]}")
    print(f"  Department: {employee_info['department'].values[0]}")
    print(f"  Normal Hours: {employee_info['normal_start'].values[0]} - {employee_info['normal_end'].values[0]}")
    print(f"  Permitted Resources: {employee_info['permitted_resource_groups'].values[0]}")

# ============================================
# BUILD TIMELINE
# ============================================

print("\n" + "="*80)
print("BUILDING INCIDENT TIMELINE")
print("="*80)

timeline = []

# ---- 1. USER AND ASSET BASELINE ----
if not employee_info.empty:
    timeline.append({
        'Event ID': 'CONTEXT-001',
        'Timestamp': '2026-09-11 16:54:00',
        'Evidence Source': 'Employee Directory / Asset Inventory',
        'Observed Action': f"User: {employee_info['employee_name'].values[0]} - Role: {employee_info['role'].values[0]}",
        'Analytical Interpretation': f"Baseline - Normal hours: 07:30-17:00. Permitted: {employee_info['permitted_resource_groups'].values[0]}. CUSTOMER_EXPORT is RESTRICTED."
    })

# ============================================
# 2. EMAIL EVENTS - PHISHING (E00305, E00307)
# ============================================

print("\n📧 Searching for phishing emails...")

# Find phishing emails to mhaingura from ofservices
phishing_emails = email_logs[
    (email_logs['recipient'] == 'mhaingura') &
    (email_logs['sender'].str.contains('ofservices', case=False, na=False))
]

print(f"Found {len(phishing_emails)} phishing emails")

for _, row in phishing_emails.iterrows():
    ts_str = row['timestamp'].strftime('%Y-%m-%d %H:%M:%S')

    # Check threat intelligence match
    ti_match = threat_intel[threat_intel['indicator'].str.contains('ofservices', case=False)] if 'threat_intel' in locals() and len(threat_intel) > 0 else pd.DataFrame()
    ti_info = f"TI: {ti_match.iloc[0]['indicator']} ({ti_match.iloc[0]['confidence']})" if not ti_match.empty else ""

    timeline.append({
        'Event ID': row['event_id'],
        'Timestamp': ts_str,
        'Evidence Source': 'Email Gateway',
        'Observed Action': f"Phishing email from {row['sender']}: {row['subject'][:45]}",
        'Analytical Interpretation': f"Initial access - URL reputation: {row['url_reputation']} - Verdict: {row['verdict']} - {ti_info}"
    })

# ============================================
# 3. C2 COMMUNICATION (Proxy Logs)
# ============================================

if 'proxy_logs' in locals() and len(proxy_logs) > 0:
    c2_events = proxy_logs[(proxy_logs['user_id'] == 'mhaingura') &
                           (proxy_logs['destination_domain'].str.contains('ofservices|sync-statistics', case=False, na=False))]

    for _, row in c2_events.iterrows():
        ts_str = row['timestamp'].strftime('%Y-%m-%d %H:%M:%S')

        # Check threat intelligence
        ti_match = threat_intel[threat_intel['indicator'] == row['destination_ip']] if 'threat_intel' in locals() and len(threat_intel) > 0 else pd.DataFrame()
        ti_info = f"TI: {ti_match.iloc[0]['indicator']} ({ti_match.iloc[0]['confidence']})" if not ti_match.empty else ""

        timeline.append({
            'Event ID': row['event_id'],
            'Timestamp': ts_str,
            'Evidence Source': 'Proxy Logs',
            'Observed Action': f"Connection to {row['destination_domain']} ({row['destination_ip']})",
            'Analytical Interpretation': f"Potential C2 communication - {ti_info}"
        })

# ============================================
# 4. MALICIOUS PROCESS EXECUTION
# ============================================

if 'process_logs' in locals() and len(process_logs) > 0:
    proc_names = ['mshta.exe', 'powershell.exe', 'rundll32.exe']
    proc_events = process_logs[(process_logs['user_id'] == 'mhaingura') &
                               (process_logs['process_name'].isin(proc_names))]

    for _, row in proc_events.iterrows():
        ts_str = row['timestamp'].strftime('%Y-%m-%d %H:%M:%S')

        if 'mshta' in row['process_name'].lower():
            interp = f"mshta.exe executed - Potential HTA download - Unsigned - Risk: {row['risk_level']}"
        elif 'EncodedCommand' in str(row['command_line']):
            interp = f"PowerShell with EncodedCommand - Obfuscation - Risk: {row['risk_level']}"
        elif 'rundll32' in row['process_name'].lower():
            interp = f"rundll32.exe loading DLL from Public folder - Risk: {row['risk_level']}"
        else:
            interp = f"Suspicious process - Risk: {row['risk_level']}"

        timeline.append({
            'Event ID': row['event_id'],
            'Timestamp': ts_str,
            'Evidence Source': 'Endpoint Logs',
            'Observed Action': f"{row['process_name']} executed by {row['parent_process']}",
            'Analytical Interpretation': interp
        })

# ============================================
# 5. VPN EVENTS
# ============================================

if 'vpn_logs' in locals() and len(vpn_logs) > 0:
    vpn_mh = vpn_logs[vpn_logs['user_id'] == 'mhaingura']

    for _, row in vpn_mh.iterrows():
        ts_str = row['timestamp'].strftime('%Y-%m-%d %H:%M:%S')

        if row['country'] == 'ZA':
            if row['result'] == 'FAILURE':
                interp = f"Failed VPN attempts from South Africa - Potential credential testing - New device"
            else:
                interp = f"Successful VPN from South Africa - Credential use from new location - MFA approved"
        else:
            interp = f"VPN access from Namibia - Normal access pattern"

        timeline.append({
            'Event ID': row['event_id'],
            'Timestamp': ts_str,
            'Evidence Source': 'VPN Logs',
            'Observed Action': f"VPN from {row['country']} on {row['device_id']} - {row['result']} - MFA: {row['mfa_result']}",
            'Analytical Interpretation': interp
        })

# ============================================
# 6. RECONNAISSANCE
# ============================================

if 'process_logs' in locals() and len(process_logs) > 0:
    recon_procs = ['net.exe', 'nltest.exe']
    recon_events = process_logs[(process_logs['user_id'] == 'mhaingura') &
                                (process_logs['process_name'].isin(recon_procs))]

    for _, row in recon_events.iterrows():
        ts_str = row['timestamp'].strftime('%Y-%m-%d %H:%M:%S')

        if 'net.exe' in row['process_name'].lower():
            interp = "Active Directory reconnaissance - Querying Domain Admins group"
        else:
            interp = "Domain controller enumeration - Discovering domain controllers"

        timeline.append({
            'Event ID': row['event_id'],
            'Timestamp': ts_str,
            'Evidence Source': 'Endpoint Logs',
            'Observed Action': f"{row['process_name']} executed",
            'Analytical Interpretation': interp
        })

# ============================================
# 7. DATA COLLECTION - Bulk KYC Downloads
# ============================================

if 'file_logs' in locals() and len(file_logs) > 0:
    kyc_downloads = file_logs[(file_logs['user_id'] == 'mhaingura') &
                              (file_logs['resource_group'] == 'CUSTOMER_EXPORT') &
                              (file_logs['action'] == 'DOWNLOAD')]

    if not kyc_downloads.empty:
        first = kyc_downloads.iloc[0]
        total_bytes = kyc_downloads['bytes_transferred'].sum()
        total_mb = total_bytes / (1024 * 1024)

        ts_str = first['timestamp'].strftime('%Y-%m-%d %H:%M:%S')

        timeline.append({
            'Event ID': f"X00760-X00795",
            'Timestamp': ts_str,
            'Evidence Source': 'File Access Logs',
            'Observed Action': f"Bulk download: {len(kyc_downloads)} files - {total_mb:.1f} MB from KYC export",
            'Analytical Interpretation': f"Access to CUSTOMER_EXPORT - User not authorised for this restricted resource"
        })

# ============================================
# 8. DATA STAGING
# ============================================

if 'process_logs' in locals() and len(process_logs) > 0:
    archive = process_logs[(process_logs['user_id'] == 'mhaingura') & (process_logs['process_name'] == '7z.exe')]

    if not archive.empty:
        row = archive.iloc[0]
        ts_str = row['timestamp'].strftime('%Y-%m-%d %H:%M:%S')

        timeline.append({
            'Event ID': row['event_id'],
            'Timestamp': ts_str,
            'Evidence Source': 'Endpoint Logs',
            'Observed Action': "7z.exe archiving KYC data to kyc_q3.zip",
            'Analytical Interpretation': "Data staging - Consolidating files into single archive"
        })

# ============================================
# 9. DATA EXFILTRATION
# ============================================

if 'proxy_logs' in locals() and len(proxy_logs) > 0:
    exfil_events = proxy_logs[(proxy_logs['user_id'] == 'mhaingura') &
                              (proxy_logs['destination_domain'].str.contains('cdn-storage', case=False, na=False))]

    total_allowed = 0
    total_blocked = 0

    for _, row in exfil_events.iterrows():
        ts_str = row['timestamp'].strftime('%Y-%m-%d %H:%M:%S')
        mb = row['bytes_out'] / (1024 * 1024)

        # Check threat intelligence
        ti_match = threat_intel[threat_intel['indicator'].str.contains(row['destination_domain'], case=False)] if 'threat_intel' in locals() and len(threat_intel) > 0 else pd.DataFrame()
        ti_info = f"TI: {ti_match.iloc[0]['indicator']} ({ti_match.iloc[0]['confidence']})" if not ti_match.empty else ""

        if row['action'] == 'ALLOW':
            total_allowed += row['bytes_out']
            interp = f"Upload: {mb:.1f} MB ALLOWED - DLP triggered but action was ALLOW - {ti_info}"
        else:
            total_blocked += row['bytes_out']
            interp = f"Upload: {mb:.1f} MB BLOCKED - DLP rule CUSTOMER-PII-BULK - {ti_info}"

        timeline.append({
            'Event ID': row['event_id'],
            'Timestamp': ts_str,
            'Evidence Source': 'Proxy/DLP Logs',
            'Observed Action': f"Upload {mb:.1f} MB to {row['destination_domain']} - {row['action']}",
            'Analytical Interpretation': interp
        })

    if total_allowed > 0 or total_blocked > 0:
        timeline.append({
            'Event ID': 'EXFIL-SUMMARY',
            'Timestamp': '2026-09-12 00:44:00',
            'Evidence Source': 'Proxy/DLP Logs (Summary)',
            'Observed Action': f"Total: {total_allowed/(1024*1024):.1f} MB allowed | {total_blocked/(1024*1024):.1f} MB blocked",
            'Analytical Interpretation': f"Exfiltration summary: {total_allowed/(1024*1024):.1f} MB exfiltrated before DLP blocked subsequent attempts"
        })

# ============================================
# 10. INVESTIGATION SESSIONS
# ============================================

if 'investigation_data' in locals() and len(investigation_data) > 0:
    high_risk = investigation_data[(investigation_data['user_id'] == 'mhaingura') & (investigation_data['risk_score'] >= 50)]

    for _, row in high_risk.iterrows():
        ts_str = row['session_start'].strftime('%Y-%m-%d %H:%M:%S')

        if 'KYC' in row['session_id']:
            interp = f"Risk: {row['risk_score']} - {row['sensitive_resources']} Restricted - {row['bytes_downloaded_mb']:.1f} MB"
        elif 'EXFIL' in row['session_id']:
            interp = f"Risk: {row['risk_score']} - {row['data_upload_mb']:.1f} MB - privilege_mismatch={row['privilege_mismatch']}"
        elif 'VPN' in row['session_id']:
            interp = f"Risk: {row['risk_score']} - impossible_travel={row['impossible_travel']}, new_device={row['new_device']}"
        else:
            interp = f"Risk: {row['risk_score']} - All indicators triggered"

        timeline.append({
            'Event ID': row['session_id'],
            'Timestamp': ts_str,
            'Evidence Source': 'Investigation Data',
            'Observed Action': f"Session {row['session_id']} - Risk Score: {row['risk_score']}",
            'Analytical Interpretation': interp
        })

# ============================================
# 11. THREAT INTELLIGENCE
# ============================================

if 'threat_intel' in locals() and len(threat_intel) > 0:
    ti_matches = threat_intel[threat_intel['indicator'].str.contains('ofservices|cdn-storage|196.250.34.77|45.83.64.12|37ed5ae', case=False, na=False)]

    for _, row in ti_matches.iterrows():
        ts = row['last_updated']
        ts_str = ts.strftime('%Y-%m-%d %H:%M:%S') if hasattr(ts, 'strftime') else str(ts)

        timeline.append({
            'Event ID': row['intel_id'],
            'Timestamp': ts_str,
            'Evidence Source': 'Threat Intelligence',
            'Observed Action': f"Indicator: {row['indicator']} - Confidence: {row['confidence']}",
            'Analytical Interpretation': f"Threat Actor: {row['threat_actor']} - Tactic: {row['tactic']}"
        })

# ============================================
# SORT AND DISPLAY
# ============================================

timeline_df = pd.DataFrame(timeline)

try:
    timeline_df['Timestamp_sort'] = pd.to_datetime(timeline_df['Timestamp'], errors='coerce')
    timeline_df = timeline_df.sort_values('Timestamp_sort')
    timeline_df = timeline_df.drop('Timestamp_sort', axis=1)
except:
    pass

print("\n" + "="*120)
print("C1: CORRELATED INCIDENT TIMELINE")
print("="*120)
print(f"Total events: {len(timeline_df)}")
print(f"Evidence sources: {', '.join(timeline_df['Evidence Source'].unique())}")
print("="*120)
print(f"{'Event ID':<15} | {'Timestamp':<20} | {'Evidence Source':<22} | {'Observed Action':<45} | Analytical Interpretation")
print("-"*135)

for _, row in timeline_df.iterrows():
    action = str(row['Observed Action'])[:43]
    interp = str(row['Analytical Interpretation'])[:55]
    print(f"{row['Event ID']:<15} | {row['Timestamp']:<20} | {row['Evidence Source']:<22} | {action:<45} | {interp}")

# ============================================
# EXPORT
# ============================================

timeline_df.to_csv('incident_timeline_complete.csv', index=False)
print("\n✅ Timeline exported to 'incident_timeline_complete.csv'")

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Total Events: {len(timeline_df)}")
print("\nEvents by Evidence Source:")
for source in timeline_df['Evidence Source'].unique():
    count = len(timeline_df[timeline_df['Evidence Source'] == source])
    print(f"  • {source}: {count} events")

print("\n✅ Copy the timeline above into your report under C1.")

C1: CORRELATED INCIDENT TIMELINE
Email | VPN | Endpoint | File | Proxy/DLP | User | Asset | Threat Intelligence

📧 Reloading email_logs from CSV...
✅ Loaded 433 email events
Columns: ['event_id', 'timestamp', 'recipient', 'sender', 'subject', 'url', 'attachment', 'spf', 'dkim', 'dmarc', 'url_reputation', 'verdict', 'reason']
  ✓ email_logs.timestamp

🕐 Converting other timestamps...
  ✓ vpn_logs.timestamp
  ✓ process_logs.timestamp
  ✓ file_logs.timestamp
  ✓ proxy_logs.timestamp
  ✓ investigation_data.session_start
✅ Timestamps converted

👤 Getting user and asset information...
User: Martha Haingura (mhaingura)
  Role: Customer Service Officer
  Department: Customer Service
  Normal Hours: 07:30 - 17:00
  Permitted Resources: CRM_BASIC;CUSTOMER_BASIC

BUILDING INCIDENT TIMELINE

📧 Searching for phishing emails...
Found 2 phishing emails

C1: CORRELATED INCIDENT TIMELINE
Total events: 42
Evidence sources: VPN Logs, Investigation Data, Employee Directory / Asset Inventory, Email Gateway

In [ ]:
# ============================================
# PART D1 - MODEL DESIGN AND IMPLEMENTATION
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("D1: MODEL DESIGN AND IMPLEMENTATION")
print("="*80)

# ============================================
# 1. LOAD AND EXPLORE TRAINING DATA
# ============================================

print("\n📂 Loading training data...")

training_data = pd.read_csv('ofs_access_session_training.csv')

print(f"Training data shape: {training_data.shape}")
print(f"Columns: {training_data.columns.tolist()}")
print(f"\nClass distribution:")
print(training_data['label'].value_counts())
print(f"Normal (0): {len(training_data[training_data['label'] == 0])}")
print(f"Suspicious (1): {len(training_data[training_data['label'] == 1])}")
print(f"Percentage suspicious: {len(training_data[training_data['label'] == 1])/len(training_data)*100:.1f}%")

# ============================================
# 2. SELECT FEATURES (Based on Data Dictionary)
# ============================================

print("\n📊 Selecting features based on data dictionary...")

# Features selected from the data dictionary with risk rationale
feature_columns = [
    'new_device',           # From data dictionary: "1 if device has not been seen for user"
    'off_hours',            # From data dictionary: "1 if outside user normal hours"
    'failed_logins_30m',    # From data dictionary: "Failed attempts in previous 30 minutes"
    'sensitive_resources',  # From data dictionary: "Count of restricted/confidential resource groups accessed"
    'peer_deviation_score', # From data dictionary: "Deviation from peer group behaviour"
    'process_risk_score',   # From data dictionary: "0-100 process-behaviour risk score"
    'destination_risk_score' # From data dictionary: "0-100 external-destination risk score"
]

print(f"Selected features: {feature_columns}")

# Prepare feature matrix and target vector
X = training_data[feature_columns]
y = training_data['label']

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")

# ============================================
# 3. CREATE STRATIFIED 70/30 TRAIN-TEST SPLIT
# ============================================

print("\n🔀 Creating stratified 70/30 train-test split...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,  # Fixed random seed for reproducibility
    stratify=y        # Preserve class distribution
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print(f"\nTraining set class distribution:")
print(y_train.value_counts())
print(f"\nTest set class distribution:")
print(y_test.value_counts())

# ============================================
# 4. SCALE FEATURES
# ============================================

print("\n📏 Scaling features...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled successfully")

# ============================================
# 5. TRAIN RANDOM FOREST CLASSIFIER
# ============================================

print("\n🌲 Training Random Forest Classifier...")

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'  # Handle any class imbalance
)

model.fit(X_train_scaled, y_train)

print("✅ Model trained successfully")

# ============================================
# 6. FEATURE IMPORTANCE
# ============================================

print("\n📊 Feature Importance:")

feature_importance = pd.DataFrame({
    'Feature': feature_columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print(feature_importance.to_string(index=False))

# ============================================
# 7. PREDICT ON TEST SET
# ============================================

print("\n🔮 Making predictions on test set...")

y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# ============================================
# 8. EVALUATION METRICS
# ============================================

print("\n" + "="*80)
print("MODEL EVALUATION RESULTS")
print("="*80)

# Confusion Matrix
print("\n📊 Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\n  True Negatives (Correctly predicted normal): {cm[0,0]}")
print(f"  False Positives (False alarms): {cm[0,1]}")
print(f"  False Negatives (Missed attacks): {cm[1,0]}")
print(f"  True Positives (Correctly detected attacks): {cm[1,1]}")

# Individual Metrics
print("\n📊 Individual Metrics:")
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
false_negative_rate = cm[1,0] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0

print(f"  Accuracy: {accuracy:.4f}  → Percentage of all predictions that are correct")
print(f"  Precision: {precision:.4f} → When model predicts attack, how often is it correct?")
print(f"  Recall: {recall:.4f}    → How many actual attacks did the model catch?")
print(f"  F1-Score: {f1:.4f}     → Harmonic mean of precision and recall")
print(f"  False Negative Rate: {false_negative_rate:.4f} → Percentage of attacks missed")

print("\n📊 Classification Report:")
print(classification_report(y_test, y_pred))

# ============================================
# 9. SCORE INVESTIGATION DATA
# ============================================

print("\n" + "="*80)
print("SCORING INVESTIGATION DATA")
print("="*80)

# Load investigation data
investigation_data = pd.read_csv('ofs_access_session_investigation.csv')
print(f"Investigation data shape: {investigation_data.shape}")

# Use same features
X_investigation = investigation_data[feature_columns]

# Scale
X_investigation_scaled = scaler.transform(X_investigation)

# Predict probabilities
investigation_data['probability'] = model.predict_proba(X_investigation_scaled)[:, 1]
investigation_data['prediction'] = model.predict(X_investigation_scaled)

# Export top 15 high-risk sessions
def get_interpretation(prob):
    if prob >= 0.9:
        return 'Very High Risk'
    elif prob >= 0.7:
        return 'High Risk'
    elif prob >= 0.5:
        return 'Medium Risk'
    else:
        return 'Low Risk'

investigation_data['interpretation'] = investigation_data['probability'].apply(get_interpretation)

top_15 = investigation_data.nlargest(15, 'probability')

# Select columns for export
export_cols = ['session_id', 'session_start', 'user_id', 'probability', 'interpretation']
top_15[export_cols].to_csv('top_15_high_risk_sessions.csv', index=False)

print("\n✅ Top 15 high-risk sessions exported to 'top_15_high_risk_sessions.csv'")

print("\n📊 Top 15 High-Risk Sessions:")
print(top_15[['session_id', 'user_id', 'probability', 'interpretation']].to_string(index=False))

# ============================================
# 10. SAVE MODEL
# ============================================

import joblib
joblib.dump(model, 'access_risk_model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print("\n✅ Model saved as 'access_risk_model.pkl'")
print("✅ Scaler saved as 'scaler.pkl'")

print("\n" + "="*80)
print("D1 COMPLETE")
print("="*80)

D1: MODEL DESIGN AND IMPLEMENTATION

📂 Loading training data...
Training data shape: (3000, 22)
Columns: ['session_id', 'session_start', 'user_id', 'department', 'source_country', 'new_device', 'off_hours', 'failed_logins_30m', 'mfa_denials', 'mfa_approvals', 'unique_resources', 'sensitive_resources', 'denied_accesses', 'files_read', 'bytes_downloaded_mb', 'data_upload_mb', 'process_risk_score', 'destination_risk_score', 'peer_deviation_score', 'impossible_travel', 'privilege_mismatch', 'label']

Class distribution:
label
0    2557
1     443
Name: count, dtype: int64
Normal (0): 2557
Suspicious (1): 443
Percentage suspicious: 14.8%

📊 Selecting features based on data dictionary...
Selected features: ['new_device', 'off_hours', 'failed_logins_30m', 'sensitive_resources', 'peer_deviation_score', 'process_risk_score', 'destination_risk_score']

Feature matrix shape: (3000, 7)
Target vector shape: (3000,)

🔀 Creating stratified 70/30 train-test split...
Training set: 2100 samples
Test set:

In [ ]:
# ============================================
# PART D2 - EVALUATION AND INVESTIGATION SCORING
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("D2: EVALUATION AND INVESTIGATION SCORING")
print("="*80)

# ============================================
# 1. LOAD AND PREPARE DATA
# ============================================

print("\n📂 Loading data...")

# Load training data
training_data = pd.read_csv('ofs_access_session_training.csv')
investigation_data = pd.read_csv('ofs_access_session_investigation.csv')

print(f"Training data: {training_data.shape}")
print(f"Investigation data: {investigation_data.shape}")

# Features from data dictionary
feature_columns = [
    'new_device',
    'off_hours',
    'failed_logins_30m',
    'sensitive_resources',
    'peer_deviation_score',
    'process_risk_score',
    'destination_risk_score'
]

X = training_data[feature_columns]
y = training_data['label']

# ============================================
# 2. TRAIN MODEL
# ============================================

print("\n🌲 Training model...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train_scaled, y_train)
print("✅ Model trained")

# ============================================
# 3. EVALUATION METRICS (D2 Requirement)
# ============================================

print("\n" + "="*80)
print("MODEL EVALUATION METRICS")
print("="*80)

# Predictions
y_pred = model.predict(X_test_scaled)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\n📊 Confusion Matrix:")
print(f"                  Predicted Normal    Predicted Suspicious")
print(f"Actual Normal        {cm[0,0]:<15}       {cm[0,1]:<15}")
print(f"Actual Suspicious    {cm[1,0]:<15}       {cm[1,1]:<15}")

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
fnr = cm[1,0] / (cm[1,0] + cm[1,1]) if (cm[1,0] + cm[1,1]) > 0 else 0

print("\n📊 Performance Metrics:")
print(f"  Accuracy:           {accuracy:.4f}")
print(f"  Precision:          {precision:.4f}")
print(f"  Recall:             {recall:.4f}")
print(f"  F1-Score:           {f1:.4f}")
print(f"  False Negative Rate: {fnr:.4f}")

# ============================================
# 4. SCORE INVESTIGATION DATA
# ============================================

print("\n" + "="*80)
print("SCORING INVESTIGATION DATA")
print("="*80)

# Scale investigation data
X_investigation = investigation_data[feature_columns]
X_investigation_scaled = scaler.transform(X_investigation)

# Predict probabilities
investigation_data['probability'] = model.predict_proba(X_investigation_scaled)[:, 1]
investigation_data['prediction'] = model.predict(X_investigation_scaled)

# Add interpretation
def get_interpretation(prob):
    if prob >= 0.9:
        return 'Very High Risk'
    elif prob >= 0.7:
        return 'High Risk'
    elif prob >= 0.5:
        return 'Medium Risk'
    else:
        return 'Low Risk'

investigation_data['interpretation'] = investigation_data['probability'].apply(get_interpretation)

# Get top 15
top_15 = investigation_data.nlargest(15, 'probability')

# Export
export_cols = ['session_id', 'session_start', 'user_id', 'probability', 'interpretation']
top_15[export_cols].to_csv('top_15_high_risk_sessions.csv', index=False)

print("\n✅ Top 15 high-risk sessions exported to 'top_15_high_risk_sessions.csv'")

print("\n📊 Top 15 High-Risk Sessions:")
print(top_15[['session_id', 'user_id', 'probability', 'interpretation']].to_string(index=False))

# ============================================
# 5. CALCULATE RULE-BASED SCORES (D2 Comparison)
# ============================================

print("\n" + "="*80)
print("RULE-BASED SCORE COMPARISON")
print("="*80)

# Calculate rule-based score (from Part B)
def calculate_risk_score(row):
    score = 0
    score += row['impossible_travel'] * 40
    score += row['privilege_mismatch'] * 35
    score += row['off_hours'] * 15
    score += row['new_device'] * 10
    return score

investigation_data['rule_score'] = investigation_data.apply(calculate_risk_score, axis=1)

# Get top 15 by rule score
top_15_rule = investigation_data.nlargest(15, 'rule_score')

print("\n📊 Top 15 by Rule-Based Score:")
print(top_15_rule[['session_id', 'user_id', 'rule_score']].to_string(index=False))

# ============================================
# 6. COMPARE MODEL VS RULE-BASED (D2 Requirement)
# ============================================

print("\n" + "="*80)
print("MODEL vs RULE-BASED COMPARISON")
print("="*80)

# Merge top 15 model and top 15 rule for comparison
top_15_model_set = set(top_15['session_id'])
top_15_rule_set = set(top_15_rule['session_id'])

agreements = top_15_model_set.intersection(top_15_rule_set)
model_only = top_15_model_set - top_15_rule_set
rule_only = top_15_rule_set - top_15_model_set

print(f"\n📊 Comparison Results:")
print(f"  Sessions in both: {len(agreements)}")
print(f"  Model only: {len(model_only)}")
print(f"  Rule only: {len(rule_only)}")

print("\n📊 Sessions in BOTH lists (Agreements):")
for sid in agreements:
    row = investigation_data[investigation_data['session_id'] == sid].iloc[0]
    print(f"  • {sid} - {row['user_id']} - Model: {row['probability']:.3f} - Rule: {row['rule_score']}")

print("\n📊 Model ONLY (Model flagged, Rule missed):")
for sid in model_only:
    row = investigation_data[investigation_data['session_id'] == sid].iloc[0]
    print(f"  • {sid} - {row['user_id']} - Model: {row['probability']:.3f} - Rule: {row['rule_score']}")

print("\n📊 Rule ONLY (Rule flagged, Model missed):")
for sid in rule_only:
    row = investigation_data[investigation_data['session_id'] == sid].iloc[0]
    print(f"  • {sid} - {row['user_id']} - Model: {row['probability']:.3f} - Rule: {row['rule_score']}")

# ============================================
# 7. HIGH-RISK SESSIONS DETAILED VIEW
# ============================================

print("\n" + "="*80)
print("DETAILED VIEW - TOP RISK SESSIONS")
print("="*80)

# Show detailed view of top sessions
detailed_cols = ['session_id', 'user_id', 'probability', 'rule_score',
                 'new_device', 'off_hours', 'impossible_travel', 'privilege_mismatch',
                 'sensitive_resources', 'data_upload_mb']

print("\nTop 10 Sessions with Features:")
print(investigation_data.nlargest(10, 'probability')[detailed_cols].to_string(index=False))

# ============================================
# 8. SUMMARY
# ============================================

print("\n" + "="*80)
print("D2 SUMMARY")
print("="*80)
print(f"Total Investigation Sessions: {len(investigation_data)}")
print(f"High Risk (prob >= 0.7): {len(investigation_data[investigation_data['probability'] >= 0.7])}")
print(f"Medium Risk (0.5 <= prob < 0.7): {len(investigation_data[(investigation_data['probability'] >= 0.5) & (investigation_data['probability'] < 0.7)])}")
print(f"Low Risk (prob < 0.5): {len(investigation_data[investigation_data['probability'] < 0.5])}")
print(f"\nTop 15 exported to 'top_15_high_risk_sessions.csv'")
print("\n✅ D2 Complete!")

D2: EVALUATION AND INVESTIGATION SCORING

📂 Loading data...
Training data: (3000, 22)
Investigation data: (450, 21)

🌲 Training model...
✅ Model trained

MODEL EVALUATION METRICS

📊 Confusion Matrix:
                  Predicted Normal    Predicted Suspicious
Actual Normal        737                   30             
Actual Suspicious    66                    67             

📊 Performance Metrics:
  Accuracy:           0.8933
  Precision:          0.6907
  Recall:             0.5038
  F1-Score:           0.5826
  False Negative Rate: 0.4962

SCORING INVESTIGATION DATA

✅ Top 15 high-risk sessions exported to 'top_15_high_risk_sessions.csv'

📊 Top 15 High-Risk Sessions:
   session_id     user_id  probability interpretation
       S00300      ntinda     0.883361      High Risk
  CASE-KYC-01   mhaingura     0.880000      High Risk
       S00386     shapaka     0.879540      High Risk
       S00427     sihuhua     0.873021      High Risk
  CASE-VPN-01   mhaingura     0.870000      High Ris